In [1]:
EXTRA_YEARS = 0

start = "2025-01-01T00:00:00+01:00"
end = f"{2025+1}-12-31T23:59:59+01:00"

In [2]:
from IPython.core.interactiveshell import InteractiveShell

import datetime

InteractiveShell.ast_node_interactivity = "all"

# Import pandas
import pandas as pd

pd.set_option('display.width', 2000)

# Import logging and surpress warnings
import logging

logging.getLogger("neo4j").setLevel(logging.ERROR)
logging.getLogger("pd").setLevel(logging.ERROR)

# Import Path
from pathlib import Path

# Import promg
from promg.modules.db_management import DBManagement
from promg import SemanticHeader, DatasetDescriptions, OcedPg, Query

In [3]:
from util.db_helper_functions import get_db_connection, get_config

In [4]:
conf_path = Path('config.yaml')

In [5]:
config = get_config(conf_path)
db_connection = get_db_connection(config)

These are the credentials that I expect to be set for the database.
db_name: neo4j
uri: bolt://localhost:7687
password: libraries
----------------------
If you have other credentials, please change them at: config.yaml



### Clear Database

In [6]:
dataset_descriptions = DatasetDescriptions(config=config)
semantic_header = SemanticHeader.create_semantic_header(config=config)
db_manager = DBManagement(db_connection=db_connection, semantic_header=semantic_header)
data_loader = OcedPg(database_connection=db_connection,
                     dataset_descriptions=dataset_descriptions,
                     semantic_header=semantic_header)

In [7]:
db_manager.clear_db()
db_manager.set_constraints()

1it [00:03,  3.41s/it, clear_db: took 3.43 seconds]

True

2it [00:03,  1.64s/it, set_constraints: took 0.4 seconds]

In [8]:
# set_index_on_validitiy_intervals for States
db_connection.exec_query('''
                         CREATE INDEX validityIntervals FOR (st: State) ON (st.validFrom, st.validTo)
                         ''')

[]

In [9]:
start_build = datetime.datetime.now()
print(f'Started at {start_build}')
data_loader.load()

Started at 2026-03-23 16:09:10.454000
10it [00:17,  2.09s/it, _filter_nodes for EventLog: took 0.0 seconds]                     

In [10]:
data_loader.transform()

25it [00:43,  1.56s/it, _create_relations_using_record for (m ) - [:MEMBER_OF] -> (l ): took 1.16 seconds]                                                                                                                                                                                                           

In [11]:
# delete all records
db_connection.exec_query('''
    :auto
    MATCH (l:Log) - [contains:CONTAINS] -> (r:Record)
    CALL (contains) {
        DELETE contains
    } IN TRANSACTIONS
''')

# delete all records
db_connection.exec_query('''
    :auto
    MATCH (l:Log)
    CALL (l) {
        DETACH DELETE l
    } IN TRANSACTIONS
''')

# delete all records
db_connection.exec_query('''
    :auto
    MATCH (r:Record) <- [ex:EXTRACTED_FROM] - (n)
    CALL (ex) {
        DELETE ex
    } IN TRANSACTIONS
''')

# delete all records
db_connection.exec_query('''
    :auto
    MATCH (r:Record) - [iot:IS_OF_TYPE] -> (rt:RecordType)
    CALL (iot) {
        DELETE iot
    } IN TRANSACTIONS
''')

# delete all records
db_connection.exec_query('''
    :auto
    MATCH (r:Record)
    CALL (r) {
        DETACH DELETE r
    } IN TRANSACTIONS
''')

db_connection.exec_query('''
    :auto
    MATCH (rt:RecordType)
    CALL (rt) {
        DETACH DELETE rt
    } IN TRANSACTIONS
''')

[]

[]

[]

[]

[]

[]

In [12]:
# copy over member attributes
query = '''
MATCH (m:Member)
WITH m ORDER BY m.sysId, m.registrationId
WITH m.sysId as memberId, min(m.registrationId) as first_registration, collect(m) as same_member
WHERE size(same_member) > 1
UNWIND range(1, size(same_member) - 1) as i
WITH same_member[0] as full_member, same_member[i] as new_member
SET new_member += {firstName: full_member.firstName, lastName: full_member.lastName, dateOfBirth: full_member.dateOfBirth, ageCategory:full_member.ageCategory}
'''

db_connection.exec_query(query)

[]

# Add interval of events

In [13]:
add_interval_to_events = '''
MATCH (e:Event)
SET e.validFrom = e.timestamp, e.validTo = e.timestamp
'''

db_connection.exec_query(add_interval_to_events)

[]

In [14]:
add_interval_to_observes = '''
:auto
MATCH (e:Event) - [c:OBSERVES] -> (n)
CALL (e,c){
    SET c.validFrom = e.timestamp, c.validTo = e.timestamp
} IN TRANSACTIONS
'''

db_connection.exec_query(add_interval_to_observes)

[]

In [15]:
db_connection.exec_query(Query(query_str='''
                                         WITH datetime($start) as start_date, datetime($end) as end_date
                                             MATCH (l:Library)
                                         SET l.validFrom = start_date
                                         SET l.validTo = end_date
                                         ''',
                               parameters={
                                   "start": start,
                                   "end": end
                               }))

[]

# Add Intervals to `Member` Object

We'll go through each object type and relationship type in our process and look at the Oracle.
Let's start with 'Member'.

- f<sub>Member</sub>(Register Member) = Create and f<sub>MEMBER_OF</sub>(Register Member) = Create
- f<sub>Member</sub>(Deregister Member) = Delete and f<sub>MEMBER_OF</sub>(Deregister Member) = Delete

- f<sub>Member</sub>(Borrow book) = Read
- f<sub>Member</sub>(Extend book) = Read
- f<sub>Member</sub>(Return book) = Read

- f<sub>Member</sub>(Reserve Book) = Read
- f<sub>Member</sub>(Increase fine) = Update (fine attribute)
- f<sub>Member</sub>(Pay fine) = Update (fine attribute)

In [16]:
db_connection.exec_query('''
    MATCH (m:Member) <- [:OBSERVES] - (e:Event)
    WHERE e.activity = 'Register Member'
    SET m.validFrom = e.timestamp, m.subscriptionType = e.subscriptionType
    MERGE (m) - [:HAS_STATE] -> (st:State {validFrom: e.timestamp})
    SET st.fine = 0
''')

db_connection.exec_query('''
    MATCH (m:Member) <- [:OBSERVES] - (e:Event)
    WHERE e.activity = 'Deregister Member'
    SET m.validTo = e.timestamp
''')

db_connection.exec_query(Query(query_str='''
                                         WITH datetime($start) as start_date, datetime($end) as end_date
                                             MATCH (m:Member)
                                         SET m.validFrom = COALESCE (m.validFrom, start_date)
                                         SET m.validTo = COALESCE (m.validTo, end_date)

                                         ''',
                               parameters={
                                   "start": start,
                                   "end": end
                               }))



[]

[]

[]

In [17]:
db_connection.exec_query('''
    MATCH (m:Member) - [member_of:MEMBER_OF] -> (l:Library)
    WITH member_of, m, l,
        CASE WHEN m.validFrom >= l.validFrom THEN m.validFrom ELSE l.validFrom END as start_value, //take max(m.validFrom, l.validFrom)
        CASE WHEN m.validTo <= l.validTo THEN m.validTo ELSE l.validTo END as end_value //take min(m.validTo, l.validTo)
    SET member_of.validFrom = start_value,
        member_of.validTo = end_value
''')

[]

Set initial state

In [18]:
# first, we use the initial fine amount when the first recorded event is not Increase fine.
# Then we can use the recorded fine attribute

db_connection.exec_query('''
    MATCH (m:Member)
    WHERE NOT EXISTS ((m) - [:HAS_STATE] -> (:State))
    CALL (m) {
      MATCH (m) <- [:OBSERVES] - (e:Event)
      RETURN e ORDER BY e.timestamp LIMIT 1
    }
    WITH m, e
    WHERE e.activity <> 'Increase fine'
    MERGE (m) - [:HAS_STATE] -> (st:State {validFrom: m.validFrom})
    SET st.fine = e.fineAmount
''')

[]

In [19]:
db_connection.exec_query('''
    MATCH (m:Member)
    WHERE NOT EXISTS ((m) - [:HAS_STATE] -> (:State))
        CALL (m) {
          MATCH (m) <- [:OBSERVES] - (e:Event)
          RETURN e ORDER BY e.timestamp LIMIT 1
        }
    WITH e, m
    CALL (e, m) {
          MATCH (m) <- [:OBSERVES] - (f:Event)
          WHERE f.activity IN ['Increase fine', 'Pay fine']
          AND f <> e // find next event that is not e
          RETURN f ORDER BY f.timestamp LIMIT 1
    }
    WITH e, f, m,
    CASE f.activity
    WHEN 'Increase fine' THEN e.fineAmount - (f.fineAmount - e.fineAmount) // take current fine - difference between two consecutive increases
    WHEN 'Pay fine' THEN -0.1
    END AS initial_fine
    MERGE (m) - [:HAS_STATE] -> (st:State {validFrom: m.validFrom})
    SET st.fine = initial_fine
''')

[]

In [20]:
# then there are some members that have not interacted with the library in this year, we don't know anything about their state
db_connection.exec_query('''
    MATCH (m:Member)
    WHERE NOT EXISTS ((m) - [:OBSERVES] - (:Event))
    AND NOT EXISTS ((m) - [:HAS_STATE] -> (:State))
    MERGE (m) - [:HAS_STATE] -> (st:State {validFrom: m.validFrom})
    SET st.fine = -0.1
''')

[]

In [21]:
db_connection.exec_query('''
    :auto
    MATCH (m:Member) <- [:OBSERVES] - (e:Event)
    WHERE e.activity IN ['Pay fine', 'Increase fine']
    CALL (m, e) {
        MERGE (m) - [:HAS_STATE] -> (st:State {validFrom: e.timestamp})
        SET st.fine = e.fineAmount
    } IN TRANSACTIONS
''')

[]

# complete states `Member`

In [22]:
order_states = '''
:auto
MATCH (m:Member) - [:HAS_STATE] -> (st:State)
WITH m, st ORDER BY st.validFrom
WITH m, collect(st) as member_states
UNWIND range(0, size(member_states) - 2) as index
WITH member_states[index] as first, member_states[index+1] as second
CALL (first, second) {
    MERGE (first) - [:NEXT] -> (second)
} IN TRANSACTIONS'''

db_connection.exec_query(order_states)

[]

In [23]:
copy_over_properties = '''
    MATCH (m:Member) - [:HAS_STATE] ->  (st:State)
    WHERE st[$prop] is NULL
    MATCH (prev) - [:NEXT] -> (st)
    SET st[$prop] = prev[$prop]
    RETURN count(st) as count
'''

properties = ['fine']
for prop in properties:
    count = 1

    while count > 0:
        result = db_connection.exec_query(query=Query(copy_over_properties,
                                                      parameters={"prop": prop}))
        count = result[0]['count']
        print(count)


0


In [24]:
determine_to_state = '''
    MATCH (m:Member) - [:HAS_STATE] ->  (st:State)
    MATCH (st) - [:NEXT] -> (successor)
    SET st.validTo = successor.validFrom
'''

db_connection.exec_query(determine_to_state)

determine_to_end_state = '''
    MATCH (m:Member) - [:HAS_STATE] ->  (st:State)
    WHERE NOT EXISTS((st) - [:NEXT] -> (:State))
    SET st.validTo = m.validTo
'''

db_connection.exec_query(determine_to_end_state)

[]

[]

In [25]:
# check whether two following states have different properties, if result is 0 then they all have different properties
check_following_states = '''
MATCH (m:Member) - [:HAS_STATE] -> (st1:State) - [:NEXT] -> (st2:State)
WHERE st1.fine = st2.fine
RETURN count(m) as member_with_same_states
'''

db_connection.exec_query(check_following_states)

[{'member_with_same_states': 129}]

# Add Intervals to `Book` Object

We'll go through each object type and relationship type in our process and look at the Oracle.
Let's start with 'Book'.
First of all, `Book` has a snapshot cardinality many-to-1 relationship with `Library` (a book is in the Catalog of one library).
So that implies that whenever we create or delete a book, we should create and delete the `IN_CATALOG` relationship.


- f<sub>Book</sub>(Add Book To Catalogue) = Create and f<sub>IN_CATALOG</sub>(Add Book To Catalogue) = Create
- f<sub>Book</sub>(Remove Book From Catalogue) = Delete and f<sub>IN_CATALOG</sub>(Remove Book From Catalogue) = Delete

- f<sub>Book</sub>(Borrow Book) = Update (available feature and Reserved feature) and f<sub>BORROWED_BY</sub>(Borrow Book) = Create
- f<sub>Book</sub>(Extend Book) = Read and f<sub>BORROWED_BY</sub>(Borrow Book) = Update (Extend count)
- f<sub>Book</sub>(Return Book) = Update (available feature) and f<sub>BORROWED_BY</sub>(Borrow Book) = Delete

- f<sub>Book</sub>(Reserve Book) = Update (reserved feature)
- f<sub>Book</sub>(Remove Reservation) = Update (reserved feature)

### f<sub>Book</sub>(Add Book To Catalogue) = Create and f<sub>IN_CATALOG</sub>(Add Book To Catalogue) = Create
First, we add the interval for the book object and the IN_CATALOG relationship.
When a book is created, we can also create the initial state for the book. We know that the book has the following non-static properties: _available_ and _reserved_

In [26]:
add_book_to_catalogue = '''
    MATCH (b:Book) - [:OBSERVES] - (e:Event)
    MATCH (b:Book) <- [ic:IN_CATALOG] - (l:Library)
    WHERE e.activity = 'Add Book To Catalogue'
    SET b.validFrom = e.timestamp, ic.validFrom = e.timestamp
    MERGE (b) - [:HAS_STATE] -> (state:State {validFrom: e.timestamp})
    SET state.available = True, state.reserved=False
'''

db_connection.exec_query(add_book_to_catalogue);

 ### f<sub>Book</sub>(Remove Book From Catalogue) = Delete and f<sub>IN_CATALOG</sub>(Remove Book From Catalogue) = Delete

In [27]:
remove_book_from_catalogue = '''
    MATCH (b:Book) - [:OBSERVES] - (e:Event)
    MATCH (b:Book) <- [ic:IN_CATALOG] - (l:Library)
    WHERE e.activity = 'Remove Book from Catalogue'
    SET b.validTo = e.timestamp, ic.validTo = e.timestamp
'''

db_connection.exec_query(remove_book_from_catalogue);

We know when books are created and deleted. If no Create event is recorded, then the Book existed from the start of Omega and if no Delete then until the end of Omega.

In [28]:
add_start_and_end = '''
                    WITH datetime($start) as start_date, datetime($end) as end_date
                        MATCH (b:Book) <- [ic:IN_CATALOG] - (l:Library)
                    SET b.validFrom = COALESCE (b.validFrom, start_date), ic.validFrom = COALESCE (ic.validFrom, start_date), b.validTo = COALESCE (b.validTo, end_date), ic.validTo = COALESCE (ic.validTo, end_date)
                    '''

query = Query(query_str=add_start_and_end,
              parameters={
                  "start": start,
                  "end": end
              })
db_connection.exec_query(query)

[]

In [29]:
add_initial_state_when_unknown = '''
    MATCH (b:Book)
    WHERE NOT EXISTS ((b) - [:HAS_STATE] -> (:State))
    OPTIONAL CALL (b) {
        MATCH (b) - [:OBSERVES] - (e:Event)
        WHERE e.activity IN ['Borrow book', 'Extend book', 'Return book', 'Remove from Catalogue']
        WITH e ORDER BY e.timestamp LIMIT 1
        RETURN CASE e.activity
        WHEN "Borrow book" THEN True
        WHEN "Extend book" THEN False
        WHEN "Return book" THEN False
        WHEN "Remove from Catalogue" THEN True
        END AS available_before_event, True as been_borrowed
    }
    OPTIONAL CALL (b) {
        MATCH (b) - [:OBSERVES] - (e:Event)
        WHERE e.activity IN ['Reserve book', 'Remove Reservation']
        WITH e ORDER BY e.timestamp LIMIT 1
        RETURN CASE e.activity
        WHEN "Reserve book" THEN False
        WHEN "Remove Reservation" THEN True
        END AS reserved_before_event
    }

    MERGE (b) - [:HAS_STATE] -> (state:State {validFrom: b.validFrom})
    SET state.available = COALESCE(available_before_event, 'unknown'), state.reserved= coalesce(reserved_before_event, not been_borrowed, 'unknown')
'''

db_connection.exec_query(add_initial_state_when_unknown)

[]

### f<sub>Book</sub>(Borrow Book) = Update (available feature)

Borrow Book also creates a BORROWED_BY relationship
f<sub>BORROWED_BY</sub>(Borrow Book) = CREATE

In [30]:
update_borrow_book_state_query = '''
    MATCH (b:Book) <- [:OBSERVES] - (e:Event)
    MATCH (m:Member) <- [:OBSERVES] - (e:Event)
    WHERE e.activity = 'Borrow book'
    MERGE (b) - [:HAS_STATE] -> (state:State {validFrom: e.timestamp})
    SET state.available = e.bookAvailable
    WITH b, m, e ORDER BY e.timestamp
    WITH b, m, collect(e) as borrowed_events
    UNWIND range(0, size(borrowed_events) - 1) as i
    WITH b, m, borrowed_events[i] as e, i as index
    MERGE (b) - [:BORROWED_BY {validFrom: e.timestamp, relIndex: index, extendCount: 0}] ->  (m)
'''

db_connection.exec_query(update_borrow_book_state_query)

[]

### f<sub>Book</sub>(Extend Book) = Read
### f<sub>Borrowed By</sub>(Extend Book) = Update

Extend Book updates the BORROWED_BY relationship
f<sub>BORROWED_BY</sub>(Borrow Book) = Update

In [31]:
update_extend_book_state_query = '''
    MATCH (b:Book) <- [:OBSERVES] - (e:Event)
    MATCH (m:Member) <- [:OBSERVES] - (e:Event)
    WHERE e.activity = 'Extend book'
    CALL (m, b, e) {
        MATCH (b) <- [:OBSERVES] - (f:Event)
        MATCH (m) <- [:OBSERVES] - (f)
        MATCH (b) - [bor:BORROWED_BY {validFrom: f.timestamp}] ->  (m)
        WHERE f.activity = 'Borrow book' and f.timestamp <= e.timestamp
        RETURN f, bor.relIndex as relIndex ORDER BY f.timestamp DESC LIMIT 1
    }
    CALL (m, b, e, f) {
        MATCH (b) <- [:OBSERVES] - (g:Event)
        MATCH (m) <- [:OBSERVES] - (g)
        WHERE g.activity = 'Extend book' AND f.timestamp <= g.timestamp <= e.timestamp
        RETURN count(distinct g) as extendCount
    }

    MERGE (b) - [:BORROWED_BY {validFrom: e.timestamp, extendCount: extendCount, relIndex: relIndex}] ->  (m)
'''

db_connection.exec_query(update_extend_book_state_query)

[]

No Previous Borrow Book has been recorded (then previous query would not return anything)

In [32]:
extend_book_query_unknown_3 = '''
MATCH (b:Book) <- [:OBSERVES] - (e:Event)
MATCH (m:Member) <- [:OBSERVES] - (e:Event)
WHERE e.activity = 'Extend book' AND NOT EXISTS ((b) - [:BORROWED_BY {validFrom: e.timestamp}] -> (m)) //no borrowed relationship exists yet
MERGE (b) - [:BORROWED_BY {validFrom: datetime($start), extendCount: 0, relIndex: -1}] ->  (m)
WITH b, m, e ORDER BY e.timestamp
WITH b, m, collect(e) as extend_events
WHERE size(extend_events) = 3 // 3 is max number of Extensions, so we know the exact extendCount
UNWIND range(0, size(extend_events) - 1) as i
WITH b, m, extend_events[i] as event, i + 1 as extendCount
MERGE (b) - [:BORROWED_BY {validFrom: event.timestamp, extendCount: extendCount, relIndex: -1}] ->  (m) //assumption, first recorded extend event is the first extend event (probably not always true)
'''

db_connection.exec_query(Query(query_str=extend_book_query_unknown_3,
                               parameters={
                                   "start": start,
                                   "end": end
                               }))

[]

In [33]:
extend_book_query_unknown_borrow = '''
MATCH (b:Book) <- [:OBSERVES] - (e:Event)
MATCH (m:Member) <- [:OBSERVES] - (e:Event)
WHERE e.activity = 'Extend book' AND NOT EXISTS ((b) - [:BORROWED_BY {validFrom: e.timestamp}] -> (m)) //no borrowed relationship exists yet
MERGE (b) - [:BORROWED_BY {validFrom: datetime($start), extendCount: 'unknown', relIndex: -1}] ->  (m)

WITH b, m, e ORDER BY e.timestamp
WITH b, m, collect(e) as extend_events
WHERE size(extend_events) < 3
UNWIND range(0, size(extend_events) - 1) as i
WITH b, m, extend_events[i] as event, 'unknown + ' + i + 1 as extendCount
MERGE (b) - [:BORROWED_BY {validFrom: event.timestamp, extendCount: extendCount, relIndex: -1}] ->  (m)
'''

db_connection.exec_query(Query(query_str=extend_book_query_unknown_borrow,
                               parameters={
                                   "start": start,
                                   "end": end
                               }))

[]

### f<sub>Book</sub>(Return Book) = Update (available feature)

Return Book also deletes the BORROWED_BY relationship
f<sub>BORROWED_BY</sub>(Borrow Book) = DELETE

In [34]:
# borrowed relationship might already exist (since previous query), so we need to find the previous relationship
update_return_book_state_query = '''
    MATCH (b:Book) <- [:OBSERVES] - (e:Event)
    MATCH (m:Member) <- [:OBSERVES] - (e:Event)
    WHERE e.activity = 'Return book'
    MERGE (b) - [:HAS_STATE] -> (state:State {validFrom: e.timestamp})
    SET state.available = e.bookAvailable
    WITH m, b, e
    CALL (m, b, e) {
        MATCH (b) - [bor:BORROWED_BY] -> (m)
        WHERE bor.validFrom <= e.timestamp
        RETURN bor ORDER BY bor.validFrom DESC LIMIT 1
    }
    SET bor.validTo = e.timestamp
'''

db_connection.exec_query(update_return_book_state_query)

[]

In [35]:
# borrowed relationship does not yet exist (can only hold for the first return book event for that (m, b) pair
update_return_book_state_query = '''
    MATCH (b:Book) <- [:OBSERVES] - (e:Event)
    MATCH (m:Member) <- [:OBSERVES] - (e:Event)
    WHERE e.activity = 'Return book' AND NOT EXISTS ((b) - [:BORROWED_BY {validTo: e.timestamp}] -> (m)) //no borrowed relationship exists yet
    MERGE (b) - [:BORROWED_BY {validTo: e.timestamp, relIndex: -1, extendCount: 'unknown'}] -> (m)
'''

db_connection.exec_query(update_return_book_state_query)

[]

### f<sub>Book</sub>(Reserve Book) = Update (reserved feature)
Reserve Book also creates a RESERVED_BY relationship
f<sub>RESERVED_BY</sub>(Borrow Book) = CREATE

In [36]:
update_reserve_book_state_query = '''
    MATCH (b:Book) <- [:OBSERVES] - (e:Event)
    MATCH (m:Member) <- [:OBSERVES] - (e:Event)
    WHERE e.activity = 'Reserve book'
    MERGE (b) - [:HAS_STATE] -> (state:State {validFrom: e.timestamp})
    SET state.reserved = e.bookReserved
    WITH b, m, e ORDER BY e.timestamp
    WITH b, m, collect(e) as reserved_events
    UNWIND range(0, size(reserved_events) - 1) as i
    WITH b, m, reserved_events[i] as e, i as index
    MERGE (b) - [:RESERVED_BY {validFrom: e.timestamp, relIndex: index}] ->  (m)
'''

db_connection.exec_query(update_reserve_book_state_query)

[]

### f<sub>Book</sub>(Remove Reservation) = Update (reserved feature)

In [37]:
# borrowed relationship might already exist (since previous query), so we need to find the previous relationship
update_remove_reservation_state_query = '''
    MATCH (b:Book) <- [:OBSERVES] - (e:Event)
    MATCH (m:Member) <- [:OBSERVES] - (e:Event)
    WHERE e.activity = 'Remove Reservation'
    MERGE (b) - [:HAS_STATE] -> (state:State {validFrom: e.timestamp})
    SET state.reserved = e.bookReserved
    WITH m, b, e
    CALL (m, b, e) {
        MATCH (b) - [res:RESERVED_BY] -> (m)
        WHERE res.validFrom <= e.timestamp
        RETURN res ORDER BY res.validFrom DESC LIMIT 1
    }
    SET res.validTo = e.timestamp
'''

db_connection.exec_query(update_remove_reservation_state_query)

[]

In [38]:
# reserved by relationship does not yet exist (can only hold for the first remove reservation event for that book
update_remove_reservation_state_query = '''
    MATCH (b:Book) <- [:OBSERVES] - (e:Event)
    MATCH (m:Member) <- [:OBSERVES] - (e:Event)
    WHERE e.activity = 'Remove Reservation' AND NOT EXISTS ((b) - [:RESERVED_BY {validTo: e.timestamp}] -> (m)) //no reserved_by relationship exists yet
    MERGE (b) - [:RESERVED_BY {validTo: e.timestamp, relIndex: -1}] -> (m)
'''

db_connection.exec_query(update_remove_reservation_state_query)

[]

# complete states book

In [39]:
order_states = '''
MATCH (b:Book) - [:HAS_STATE] -> (st:State)
WITH b, st ORDER BY st.validFrom
WITH b, collect(st) as book_states
UNWIND range(0, size(book_states) - 2) as index
WITH book_states[index] as first, book_states[index+1] as second
MERGE (first) - [:NEXT] -> (second)'''

db_connection.exec_query(order_states)

[]

In [40]:
copy_over_properties = '''
    MATCH (b:Book) - [:HAS_STATE] ->  (st:State)
    WHERE st[$prop] is NULL
    MATCH (prev) - [:NEXT] -> (st)
    SET st[$prop] = prev[$prop]
    RETURN count(st) as count
'''

properties = ['reserved', 'available']
for prop in properties:
    count = 1
    while count > 0:
        result = db_connection.exec_query(query=Query(copy_over_properties,
                                                      parameters={"prop": prop}))
        count = result[0]['count']


In [41]:
determine_to_state = '''
    MATCH (b:Book) - [:HAS_STATE] ->  (st:State)
    MATCH (st) - [:NEXT] -> (successor)
    SET st.validTo = successor.validFrom
'''

db_connection.exec_query(determine_to_state)

determine_to_end_state = '''
    MATCH (b:Book) - [:HAS_STATE] ->  (st:State)
    WHERE NOT EXISTS((st) - [:NEXT] -> (:State))
    SET st.validTo = b.validTo
'''

db_connection.exec_query(determine_to_end_state)

[]

[]

In [42]:
# check whether two following states have different properties, if result is 0 then they all have different properties
check_following_states = '''
MATCH (b:Book) - [:HAS_STATE] -> (st1:State) - [:NEXT] -> (st2:State)
WHERE st1.available = st2.available AND st1.reserved = st2.reserved
RETURN count(b) as book_with_same_states
'''

db_connection.exec_query(check_following_states)

[{'book_with_same_states': 115}]

# Complete BORROWED_BY relationship

In [43]:
db_connection.exec_query(Query(query_str='''
                                         WITH datetime($start) as start_date, datetime($end) as end_date
                                             MATCH (b:Book) - [bor:BORROWED_BY] -> (m:Member)
                                         SET bor.validFrom = COALESCE (bor.validFrom, start_date)
                                         ''',
                               parameters={
                                   "start": start,
                                   "end": end
                               }))

[]

In [44]:
db_connection.exec_query('''
    MATCH (b:Book) - [bor:BORROWED_BY] -> (m:Member)
   WITH b, m, bor.relIndex as relIndex, bor ORDER BY bor.validFrom
   WITH b, m, relIndex, collect(bor) as same_rels
   UNWIND range(0, size(same_rels) - 2) as index
   WITH same_rels[index] as first, same_rels[index+1] as second
   SET first.validTo = second.validFrom
''')



[]

In [45]:
db_connection.exec_query(Query(query_str='''
                                         WITH datetime($start) as start_date, datetime($end) as end_date
                                             MATCH (b:Book) - [bor:BORROWED_BY] -> (m:Member)
                                         SET bor.validTo = COALESCE (bor.validTo, end_date)
                                         ''',
                               parameters={
                                   "start": start,
                                   "end": end
                               }))

[]

# Complete RESERVED_BY relationship

In [46]:
db_connection.exec_query(Query(query_str='''
                                         WITH datetime($start) as start_date, datetime($end) as end_date
                                             MATCH (b:Book) - [res:RESERVED_BY] -> (m:Member)
                                         SET res.validTo = COALESCE (res.validTo, end_date), res.validFrom = COALESCE (res.validFrom, start_date)
                                         ''',
                               parameters={
                                   "start": start,
                                   "end": end
                               }))

[]

In [47]:
# Add validity intervals to HAS_STATE

In [48]:
db_connection.exec_query(Query(query_str='''
    :auto
     MATCH (n) - [hs:HAS_STATE] -> (st:State)
     CALL (st, hs) {
     SET hs.validTo = st.validTo,
        hs.validFrom = st.validFrom
    } IN TRANSACTIONS
     '''))

[]

In [49]:
end_build = datetime.datetime.now()
print(f'Ended at {end_build}')

Ended at 2026-03-23 16:10:41.769383


In [50]:
start_constraint = datetime.datetime.now()
print(f'Constraints at {start_constraint}')

Constraints at 2026-03-23 16:10:41.796959


# Check semantic constraints for Member

In [51]:
check_members_query = '''
    :auto
    MATCH (e:Event)
    WHERE e.activity IN ['Register Member',
                         'Deregister Member',
                         'Borrow book',
                         'Extend book',
                         'Return book',
                         'Reserve book',
                         'Increase fine',
                         'Pay fine']
    CALL (e) {
     RETURN
     CASE e.activity
     WHEN 'Register Member' THEN 'CREATE (Member exists from time of this event ' + e.activity + ' onwards)'
     WHEN 'Deregister Member' THEN 'DELETE (Member exists until time of this event ' + e.activity + ')'
     WHEN 'Increase fine' THEN 'UPDATE (Member has new state from time of this event ' + e.activity + ' onwards)'
     WHEN 'Pay fine' THEN 'UPDATE (Member has new state from time of this event ' + e.activity + ' onwards)'
     ELSE 'READ (Member exists at time of this event ' + e.activity + ')'
     END AS constraint,
    CASE e.activity
     WHEN 'Register Member' THEN EXISTS ((e) - [:OBSERVES] -> (:Member {validFrom: e.timestamp}))
     WHEN 'Deregister Member' THEN EXISTS ((e) - [:OBSERVES] -> (:Member {validTo: e.timestamp}))
     WHEN 'Increase fine' THEN EXISTS {
        (e) - [:OBSERVES] -> (:Member) - [:HAS_STATE] -> (st:State {validFrom: e.timestamp}) <- [:NEXT] - (prev:State)
        WHERE any(k IN keys(prev)+keys(st) WHERE NOT k IN ['validFrom','validTo'] AND properties(st)[k] <> properties(prev)[k])
     }
     WHEN 'Pay fine' THEN EXISTS {
        (e) - [:OBSERVES] -> (:Member) - [:HAS_STATE] -> (st:State {validFrom: e.timestamp}) <- [:NEXT] - (prev:State)
        WHERE any(k IN keys(prev)+keys(st) WHERE NOT k IN ['validFrom','validTo'] AND properties(st)[k] <> properties(prev)[k])
     }
     ELSE EXISTS ((e) - [:OBSERVES] -> (:Member))

    END AS satisfied_constraint
    } IN TRANSACTIONS
    WITH constraint, satisfied_constraint, e
    WHERE satisfied_constraint = False
    RETURN constraint, satisfied_constraint, count(e) ORDER BY constraint
'''

db_connection.exec_query(check_members_query)

# if an empty list is returned [] then no violations detected

[{'constraint': 'UPDATE (Member has new state from time of this event Increase fine onwards)',
  'satisfied_constraint': False,
  'count(e)': 128},
 {'constraint': 'UPDATE (Member has new state from time of this event Pay fine onwards)',
  'satisfied_constraint': False,
  'count(e)': 1}]

# Check semantic constraints for `Book`

In [52]:
check_books_query = '''
:auto
    MATCH (e:Event)
    WHERE e.activity IN ['Add Book To Catalogue',
                         'Remove Book from Catalogue',
                         'Borrow book',
                         'Extend book',
                         'Return book',
                         'Reserve book',
                         'Remove Reservation']
    CALL (e) {
     RETURN
     CASE e.activity
     WHEN 'Add Book To Catalogue' THEN 'CREATE (Book exists from time of this event ' + e.activity + ' onwards)'
     WHEN 'Remove Book from Catalogue' THEN 'DELETE (Book exists until time of this event ' + e.activity + ')'
     WHEN 'Borrow book' THEN 'UPDATE (Book has new state from time of this event ' + e.activity + ' onwards)'
     WHEN 'Extend book' THEN 'READ (Book exists at the time of this event ' + e.activity + ')'
     WHEN 'Return book' THEN 'UPDATE (Book has new state from time of this event ' + e.activity + ' onwards)'
     WHEN 'Reserve book' THEN 'UPDATE (Book has new state from time of this event ' + e.activity + ' onwards)'
     WHEN 'Remove Reservation' THEN 'UPDATE (Book has new state from time of this event ' + e.activity + ' onwards)'
     END AS constraint,
    CASE e.activity
     WHEN 'Remove Book from Catalogue' THEN EXISTS ((e) - [:OBSERVES] -> (:Book {validTo: e.timestamp}))
     WHEN 'Add Book To Catalogue' THEN EXISTS ((e) - [:OBSERVES] -> (:Book {validFrom: e.timestamp}))
     WHEN 'Borrow book' THEN EXISTS {
        (e) - [:OBSERVES] -> (:Book) - [:HAS_STATE] -> (st:State {validFrom: e.timestamp}) <- [:NEXT] - (prev:State)
        WHERE any(k IN keys(prev)+keys(st) WHERE NOT k IN ['validFrom','validTo'] AND properties(st)[k] <> properties(prev)[k])
     }
     WHEN 'Extend book' THEN EXISTS ((e) - [:OBSERVES] -> (:Book))
     WHEN 'Return book' THEN EXISTS {
        (e) - [:OBSERVES] -> (:Book) - [:HAS_STATE] -> (st:State {validFrom: e.timestamp}) <- [:NEXT] - (prev:State)
        WHERE any(k IN keys(prev)+keys(st) WHERE NOT k IN ['validFrom','validTo'] AND properties(st)[k] <> properties(prev)[k])
     }
     WHEN 'Reserve book' THEN EXISTS {
        (e) - [:OBSERVES] -> (:Book) - [:HAS_STATE] -> (st:State {validFrom: e.timestamp}) <- [:NEXT] - (prev:State)
        WHERE any(k IN keys(prev)+keys(st) WHERE NOT k IN ['validFrom','validTo'] AND properties(st)[k] <> properties(prev)[k])
     }
     WHEN 'Remove Reservation' THEN EXISTS {
        (e) - [:OBSERVES] -> (:Book) - [:HAS_STATE] -> (st:State {validFrom: e.timestamp}) <- [:NEXT] - (prev:State)
        WHERE any(k IN keys(prev)+keys(st) WHERE NOT k IN ['validFrom','validTo'] AND properties(st)[k] <> properties(prev)[k])
     }

    END AS satisfied_constraint
    } IN TRANSACTIONS
    WITH constraint, satisfied_constraint, e
    WHERE satisfied_constraint = False
    RETURN constraint,satisfied_constraint, count(e)
'''

db_connection.exec_query(check_books_query)

# if an empty list is returned [] then no violations detected

[{'constraint': 'UPDATE (Book has new state from time of this event Borrow book onwards)',
  'satisfied_constraint': False,
  'count(e)': 26}]

# Check Semantic Constraints for `IN_CATALOG`

In [53]:
check_in_catalog_query = '''
:auto
    MATCH (e:Event)
    WHERE e.activity IN ['Add Book To Catalogue',
                         'Remove Book from Catalogue']
CALL (e) {
     RETURN
     CASE e.activity
     WHEN 'Add Book To Catalogue' THEN 'CREATE (IN_CATALOG exists from time of this event ' + e.activity + ' onwards)'
     WHEN 'Remove Book from Catalogue' THEN 'DELETE (IN_CATALOG exists until time of this event ' + e.activity + ')'
     END AS constraint,
    CASE e.activity

     WHEN 'Add Book To Catalogue' THEN EXISTS ((e) - [:OBSERVES] -> (:Book) <- [:IN_CATALOG {validFrom: e.timestamp}] - (:Library))
     WHEN 'Remove Book from Catalogue' THEN EXISTS ((e) - [:OBSERVES] -> (:Book) <- [:IN_CATALOG {validTo: e.timestamp}] - (:Library))

    END AS satisfied_constraint
    } IN TRANSACTIONS
    WITH constraint, satisfied_constraint, e
    WHERE satisfied_constraint = False
    RETURN constraint,satisfied_constraint, count(e)
'''

db_connection.exec_query(check_in_catalog_query)

# if an empty list is returned [] then no violations detected

[]

# Check Semantic Constraints for 'BORROWED_BY'

In [54]:
check_query = '''
:auto
MATCH (e:Event)
WHERE e.activity IN ['Borrow book', 'Extend book', 'Return book']
CALL (e) {
RETURN
     CASE e.activity
     WHEN 'Borrow book' THEN 'CREATE (BORROWED_BY exists from time of this event ' + e.activity + ' onwards)'
     WHEN 'Extend book' THEN 'UPDATE (BORROWED_BY is updated at the time of this event ' + e.activity + ')'
     WHEN 'Return book' THEN 'DELETE (BORROWED_BY is deleted at the time of this event ' + e.activity + ')'
     END AS constraint,

CASE e.activity
WHEN 'Borrow book' THEN  EXISTS{(e) - [:OBSERVES] -> (:Book) - [bor:BORROWED_BY] -> (:Member) WHERE bor.validFrom = e.timestamp}
WHEN 'Extend book' THEN EXISTS{(e) - [:OBSERVES] -> (b:Book) - [bor:BORROWED_BY] -> (:Member) <- [prev_bor:BORROWED_BY] - (b:Book)
    WHERE bor.validFrom = e.timestamp AND prev_bor.validTo = e.timestamp
    AND any(k IN keys(prev_bor)+keys(bor) WHERE NOT k IN ['validFrom','validTo', 'relIndex'] AND properties(prev_bor)[k] <> properties(bor)[k])
}
WHEN 'Return book' THEN EXISTS{(e) - [:OBSERVES] -> (:Book) - [bor:BORROWED_BY] -> (:Member) WHERE bor.validTo = e.timestamp}
END AS satisfied_constraint
    } IN TRANSACTIONS
    WITH constraint, satisfied_constraint, e
    WHERE satisfied_constraint = False
    RETURN constraint,satisfied_constraint, count(e)
'''

db_connection.exec_query(check_query)

# # if an empty list is returned [] then no violations detected

[]

# Check Semantic Constraints for `RESERVED_BY`

In [55]:
check_query = '''
:auto
MATCH (e:Event)
WHERE e.activity IN ['Reserve book', 'Remove Reservation']
CALL (e) {
RETURN
     CASE e.activity
     WHEN 'Reserve book' THEN 'CREATE (RESERVED_BY exists from time of this event ' + e.activity + ' onwards)'
     WHEN 'Remove Reservation' THEN 'DELETE (RESERVED_BY is deleted at the time of this event ' + e.activity + ')'
     END AS constraint,

CASE e.activity
WHEN 'Reserve book' THEN  EXISTS{(e) - [:OBSERVES] -> (:Book) - [res:RESERVED_BY] -> (:Member) WHERE res.validFrom = e.timestamp}
WHEN 'Remove Reservation' THEN EXISTS{(e) - [:OBSERVES] -> (:Book) - [res:RESERVED_BY] -> (:Member) WHERE res.validTo = e.timestamp}
END AS satisfied_constraint
    } IN TRANSACTIONS
    WITH constraint, satisfied_constraint, e
    WHERE satisfied_constraint = False
    RETURN constraint,satisfied_constraint, count(e)
'''

db_connection.exec_query(check_query)

# if an empty list is returned [] then no violations detected

[]

# Check IT-OCED Specifications

In [56]:
# Every node has a validFrom and validTo
db_connection.exec_query('''
    MATCH (n)
    RETURN n.validFrom is not null as has_start, n.validTo is not null as has_end, count(n)
''')

[{'has_start': True, 'has_end': True, 'count(n)': 333214}]

In [57]:
# All nodes with states have a state starting at start of the object, ending at end of the object and there are no states superseding and exceeding the object validity
db_connection.exec_query('''
    MATCH (n) - [:HAS_STATE] -> (state:State)
    WITH n,
        min(state.validFrom) AS min_state_start,
        max(state.validTo)   AS max_state_end
    RETURN  max_state_end = n.validTo as has_valid_start,
            min_state_start = n.validFrom as has_valid_end,
    count(n)
''')

[{'has_valid_start': True, 'has_valid_end': True, 'count(n)': 8224}]

In [58]:
# two subsequent states have no gap
db_connection.exec_query('''
    MATCH (n) - [:HAS_STATE] -> (st1:State) - [:NEXT] -> (st2:State)
    RETURN st1.validTo = st2.validFrom as no_gap, count(n)
''')

[{'no_gap': True, 'count(n)': 161288}]

In [59]:
# relation have a validity interval
db_connection.exec_query('''
    MATCH (n) - [rel] -> (n2)
    WHERE type(rel) <> 'NEXT'
    RETURN rel.validFrom is not null as has_start, rel.validTo is not null as has_end, count(rel)
''')

[{'has_start': True, 'has_end': True, 'count(rel)': 419337}]

In [60]:
# relation only holds when other sides hold
db_connection.exec_query('''
    MATCH (n) - [rel] -> (n2)
    WHERE rel.validFrom is not null and rel.validTo is not null

    WITH rel, n, n2,
        CASE WHEN n.validFrom >= n2.validFrom THEN n.validFrom ELSE n2.validFrom END AS start, //take max(n.validFrom, n2.validFrom)
        CASE WHEN n.validTo <= n2.validTo THEN n.validTo ELSE n2.validTo END AS end //take min(n.validTo, n2.validTo)

    RETURN start <= rel.validFrom, rel.validTo <= end, count(rel)
''')

[{'start <= rel.validFrom': True,
  'rel.validTo <= end': True,
  'count(rel)': 419337}]

# Check Snapshot Cardinality Constraints

## BORROWED_BY

BORROWED_BY is a binary relationship between Book and Member with the following snapshot cardinalities:

- Book → Member: 0..1 at any point in time (a book may be borrowed by at most one member simultaneously)
- Member → Book: 0..5 at any point in time (a member may borrow at most five books simultaneously)

First, we check `Book → Member: 0..1` at any point in time

In [61]:
snapshot_cardinality_0_1_check = '''
    MATCH (n:$first_label)-[r:$rel_type]-(:$second_label)
    WITH n, r ORDER BY r.validFrom
    WITH n, collect(r) AS iv
    WITH n, iv[0..-1] AS iv1, iv[1..] AS iv2
    UNWIND range(0, size(iv2)-2) AS k
    WITH n, iv1[k] AS prev, iv2[k] AS next, k, iv2
    WHERE next.validFrom < prev.validTo
    RETURN DISTINCT n.sysId
'''

db_connection.exec_query(Query(
    query_str=snapshot_cardinality_0_1_check,
    template_string_parameters={
        'rel_type': 'BORROWED_BY',
        'first_label': 'Book',
        'second_label': 'Member'
    }
))

# if an empty list is returned [] then no violations detected

[{'n.sysId': '0114688'},
 {'n.sysId': '0131306'},
 {'n.sysId': '0114680'},
 {'n.sysId': '019288'},
 {'n.sysId': '0115788'},
 {'n.sysId': '015356'},
 {'n.sysId': '0130403'},
 {'n.sysId': '0114026'},
 {'n.sysId': '0141663'},
 {'n.sysId': '015821'},
 {'n.sysId': '0111597'},
 {'n.sysId': '0123847'},
 {'n.sysId': '0121621'},
 {'n.sysId': '0113737'},
 {'n.sysId': '0131507'},
 {'n.sysId': '013301'},
 {'n.sysId': '0113733'},
 {'n.sysId': '0137594'},
 {'n.sysId': '014595'},
 {'n.sysId': '0132750'},
 {'n.sysId': '013692'},
 {'n.sysId': '0135805'},
 {'n.sysId': '0119485'},
 {'n.sysId': '0134941'},
 {'n.sysId': '0117461'},
 {'n.sysId': '012804'},
 {'n.sysId': '0124601'},
 {'n.sysId': '0121353'},
 {'n.sysId': '0124337'},
 {'n.sysId': '0113619'},
 {'n.sysId': '0125025'},
 {'n.sysId': '0127253'},
 {'n.sysId': '0127932'},
 {'n.sysId': '0138977'},
 {'n.sysId': '0117018'},
 {'n.sysId': '0138469'},
 {'n.sysId': '0118878'},
 {'n.sysId': '0128539'},
 {'n.sysId': '0137674'},
 {'n.sysId': '0141538'},
 {'n.sy

Next, we check `Member → Book: 0..5` at any point in time

In [62]:
snapshot_cardinality_0_limited_n_check = '''
    MATCH (n:$first_label)-[r:$rel_type]-(:$second_label)
    WITH n, collect(r) AS iv

    // Compare only adjacent pairs to detect any overlap,
    // but we also need peak concurrency (K>1), which requires a sweep.
    // For clarity, use the event-based solution above for true K>1 checks.
    WITH n,
         [r IN iv | {t: r.validTo, delta: -1, kind: 0}] +
         [r IN iv | {t: r.validFrom, delta: +1, kind: 1}] AS updateCounts
    UNWIND updateCounts AS update
    WITH n, update
    ORDER BY elementId(n), update.t, update.kind
    WITH n, collect(update) AS updates
    WITH n,
         reduce(acc = {cur:0, max:0}, x IN updates |
           {cur: acc.cur + x.delta,
            max: CASE WHEN acc.cur + x.delta > acc.max THEN acc.cur + x.delta ELSE acc.max END}) AS r
    WHERE r.max > $max_limit
    RETURN n.sysId, r.max AS peak
    ORDER BY peak DESC;
'''

db_connection.exec_query(Query(
    query_str=snapshot_cardinality_0_limited_n_check,
    template_string_parameters={
        'rel_type': 'BORROWED_BY',
        'first_label': 'Member',
        'second_label': 'Book',
    },
    parameters={
        'max_limit': 5
    }
))

# if an empty list is returned [] then no violations detected

[{'n.sysId': 'M029', 'peak': 9},
 {'n.sysId': 'M335', 'peak': 9},
 {'n.sysId': 'M096', 'peak': 8},
 {'n.sysId': 'M196', 'peak': 8},
 {'n.sysId': 'M306', 'peak': 8},
 {'n.sysId': 'M591', 'peak': 8},
 {'n.sysId': 'M593', 'peak': 8},
 {'n.sysId': 'M747', 'peak': 8},
 {'n.sysId': 'M788', 'peak': 8},
 {'n.sysId': 'M1023', 'peak': 8},
 {'n.sysId': 'M1175', 'peak': 8},
 {'n.sysId': 'M013', 'peak': 7},
 {'n.sysId': 'M065', 'peak': 7},
 {'n.sysId': 'M163', 'peak': 7},
 {'n.sysId': 'M165', 'peak': 7},
 {'n.sysId': 'M288', 'peak': 7},
 {'n.sysId': 'M322', 'peak': 7},
 {'n.sysId': 'M351', 'peak': 7},
 {'n.sysId': 'M367', 'peak': 7},
 {'n.sysId': 'M464', 'peak': 7},
 {'n.sysId': 'M656', 'peak': 7},
 {'n.sysId': 'M691', 'peak': 7},
 {'n.sysId': 'M828', 'peak': 7},
 {'n.sysId': 'M896', 'peak': 7},
 {'n.sysId': 'M997', 'peak': 7},
 {'n.sysId': 'M1174', 'peak': 7},
 {'n.sysId': 'M1186', 'peak': 7},
 {'n.sysId': 'M404', 'peak': 6},
 {'n.sysId': 'M522', 'peak': 6},
 {'n.sysId': 'M586', 'peak': 6},
 {'n.s

## RESERVED_BY

RESERVED_BY is a binary relationship between Book and Member with the following snapshot cardinalities:

- Book → Member: 0..1 at any point in time (a book may be reserved by at most one member simultaneously)
- Member → Book: 0..5 at any point in time (a member may reserve at most five books simultaneously)

First, we check `Book → Member: 0..1` at any point in time

In [63]:
db_connection.exec_query(Query(
    query_str=snapshot_cardinality_0_1_check,
    template_string_parameters={
        'rel_type': 'RESERVED_BY',
        'first_label': 'Book',
        'second_label': 'Member'
    }
))

# if an empty list is returned [] then no violations detected

[]

Then, we check `Member → Book: 0..5` at any point in time

In [64]:
db_connection.exec_query(Query(
    query_str=snapshot_cardinality_0_limited_n_check,
    template_string_parameters={
        'rel_type': 'RESERVED_BY',
        'first_label': 'Member',
        'second_label': 'Book',
    },
    parameters={
        'max_limit': 5
    }
))

# if an empty list is returned [] then no violations detected

[]

## IN_CATALOG

IN_CATALOG is a binary relationship between Book and Library with the following snapshot cardinalities:

- Book → Library: 1..1 at any point in time (a book should always be in the catalog of exactly one library simultaneously)
- Library → Book: 0..n at any point in time (a library may have an unlimited books in its catalog simultaneously)

First, we check `Book → Library: 1..1` at any point in time

In [65]:
snapshot_cardinality_1_1_check = '''
    MATCH (n:$first_label)-[r:$rel_type]-(:$second_label)
    WITH n, r ORDER BY r.validFrom
    WITH n, collect(r) AS iv
    WITH n, iv[0] as first, iv[0..-1] AS iv1, iv[1..] AS iv2, iv[-1] as last
    UNWIND range(0, size(iv2)-2) AS k
    WITH n, first, last, iv1[k] AS prev, iv2[k] AS next, k, iv2
    WHERE first.validFrom <> n.validFrom // starts too late or too early
          OR  n.validTo <> last.validTo // end too late or too early
          OR next.validFrom <> prev.validTo // gap or overlap
    RETURN DISTINCT n, prev, next
'''

db_connection.exec_query(Query(
    query_str=snapshot_cardinality_1_1_check,
    template_string_parameters={
        'rel_type': 'IN_CATALOG',
        'first_label': 'Book',
        'second_label': 'Library'
    }
))

# if an empty list is returned [] then no violations detected

[]

`Library → Book: 0..n` at any point in time (a library may have an unlimited books in its catalog simultaneously)
This does not require any check

## Member Of

MEMBER_OF is a binary relationship between Member and Library with the following snapshot cardinalities:

- Member → Library: 1..n at any point in time (a member should always be a member of at least one library simultaneously)
- Library → Member: 0..n at any point in time (a library may have an unlimited number of members simultaneously)



In [66]:
snapshot_cardinality_1_n_check = '''
    MATCH (n:$first_label)-[r:$rel_type]-(:$second_label)
    WITH n, r ORDER BY r.validFrom
    WITH n, collect(r) AS iv
    WITH n, iv[0] as first, iv[0..-1] AS iv1, iv[1..] AS iv2, iv[-1] as last
    UNWIND range(0, size(iv2)-2) AS k
    WITH n, first, last, iv1[k] AS prev, iv2[k] AS next, k, iv2
    WHERE first.validFrom <> n.validFrom // starts too late or too early
          OR  n.validTo <> last.validTo // end too late or too early
          OR next.validFrom > prev.validTo // gap between intervals
    RETURN DISTINCT n, prev, next
'''

db_connection.exec_query(Query(
    query_str=snapshot_cardinality_1_n_check,
    template_string_parameters={
        'rel_type': 'MEMBER_OF',
        'first_label': 'Member',
        'second_label': 'Library'
    }
))

# if an empty list is returned [] then no violations detected

[]

# Lifetime Constraints

In our library, a book can be borrowed over its lifetime at most 10 times.

In [67]:
lifetime_cardinality_max_check = '''
    MATCH (n:$first_label)-[r:$rel_type]-(n2:$second_label)
    WITH n, n2, count(distinct r.relIndex) as num_rel_between
    WITH n, sum(num_rel_between) as lifetime_rel
    WHERE NOT ($min_lifetime <= lifetime_rel <= $max_lifetime)
    RETURN distinct n.sysId as violating_constraint, lifetime_rel
'''

db_connection.exec_query(Query(
    query_str=lifetime_cardinality_max_check,
    template_string_parameters={
        'rel_type': 'BORROWED_BY',
        'first_label': 'Book',
        'second_label': 'Member',
        'min_lifetime': 0,
        'max_lifetime': 10
    }
))

# if an empty list is returned [] then no violations detected

[]

In [68]:
end_constraint = datetime.datetime.now()
print(f'Constraints finished at {end_constraint}')

Constraints finished at 2026-03-23 16:11:02.543431


# Print some statistics

In [69]:
building_time = end_build - start_build
constraint_checking = end_constraint - start_constraint
print(f'Building time: {building_time}')
print(f'Constraint Checking time: {constraint_checking}')

Building time: 0:01:31.315383
Constraint Checking time: 0:00:20.746472


In [70]:
db_connection.exec_query('''
    MATCH (e:Event)
    RETURN count(e) as eventCount
''')

[{'eventCount': 155477}]

In [71]:
db_connection.exec_query('''
    MATCH (m:Member)
    RETURN count(m) as memberCount
''')

[{'memberCount': 1224}]

In [72]:
db_connection.exec_query('''
    MATCH (b:Book)
    RETURN count(b) as bookCount
''')

[{'bookCount': 7000}]

In [73]:
db_connection.exec_query('''
:auto
    MATCH (n) - [rel] -> (n2)
    WHERE type(rel) <> 'NEXT' AND type(rel) <> 'HAS_STATE'
    CALL (n, rel, n2) {
        WITH type(rel) as type, n, n2, rel.relIndex as idx, count(rel) as count
        RETURN type, count
    } IN TRANSACTIONS
    RETURN type = 'OBSERVES' as observed, sum(count)
''')

[{'observed': False, 'sum(count)': 43445},
 {'observed': True, 'sum(count)': 206380}]